# YouTube Two-Level Recommendation Scraper

For each seed video, collects its top-5 related videos (first level), then for each first-level video, collects its own top-5 related videos (second level). Uses Selenium in incognito mode; Chrome is fully restarted with a fresh temp profile before every single video visit to prevent within-session personalization drift.

Input: `seed_vid_list.csv` (columns: `link`, `stance`)

In [ ]:
# !pip install selenium pandas webdriver-manager tqdm --quiet

In [48]:
import os
import time
import random
import tempfile
import shutil
import pandas as pd
from urllib.parse import urlparse, parse_qs
from tqdm.notebook import tqdm

from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.common.exceptions import TimeoutException, NoSuchElementException, WebDriverException

try:
    from webdriver_manager.chrome import ChromeDriverManager
    USE_WDM = True
    print("webdriver-manager available — ChromeDriver 会自动安装。")
except ImportError:
    USE_WDM = False
    print("webdriver-manager 未安装 — 将使用系统自带的 ChromeDriver。")

print("Libraries imported successfully.")

webdriver-manager available — ChromeDriver 会自动安装。
Libraries imported successfully.


In [49]:
TOP_N = 5                     # related videos to collect per level
MIN_DELAY, MAX_DELAY = 8, 15  # random delay between page visits (seconds)
PAGE_TIMEOUT = 25              # page load timeout (seconds)

INPUT_FILE = "seed_vid_list_4.csv"

OUTPUT_DIR = "restart_recomm_4"
OUTPUT_FILE = os.path.join(OUTPUT_DIR, "restart_recom_4.csv")
CHECKPOINT_L1 = os.path.join(OUTPUT_DIR, "_checkpoint_level1.csv")
CHECKPOINT_L2 = os.path.join(OUTPUT_DIR, "_checkpoint_level2.csv")
os.makedirs(OUTPUT_DIR, exist_ok=True)

print(f"TOP_N = {TOP_N}")
print(f"delay = {MIN_DELAY}-{MAX_DELAY}s")
print(f"input  = {INPUT_FILE}")
print(f"output = {OUTPUT_FILE}")

TOP_N = 5
delay = 8-15s
input  = seed_vid_list_4.csv
output = restart_recomm_4/restart_recom_4.csv


In [50]:
def extract_video_id(url):
    """Extract the 11-char video_id from watch/youtu.be/shorts URLs."""
    if not isinstance(url, str):
        return None
    parsed = urlparse(url.strip())
    if "youtu.be" in parsed.netloc:
        return parsed.path.lstrip("/").split("?")[0]
    if "/shorts/" in parsed.path:
        return parsed.path.split("/shorts/")[-1].split("?")[0]
    qs = parse_qs(parsed.query)
    return qs["v"][0] if "v" in qs else None


seed_df = pd.read_csv(INPUT_FILE)
seed_df["video_id"] = seed_df["link"].apply(extract_video_id)

n_before = len(seed_df)
seed_df = seed_df.dropna(subset=["video_id"]).reset_index(drop=True)
n_after = len(seed_df)

if n_after < n_before:
    print(f"Warning: {n_before - n_after} links failed to parse into video_id and were dropped.")

print(f"共加载 {n_after} 个 seed videos")
print(seed_df["stance"].value_counts())
seed_df.head()

共加载 5 个 seed videos
stance
Mainstream    3
Right         2
Name: count, dtype: int64


,link,stance,video_id
0,https://www.youtube.com/watch?v=HgrQsU8TMmQ,Mainstream,HgrQsU8TMmQ
1,https://www.youtube.com/watch?v=A-zFK_LPxJI&t=1s,Mainstream,A-zFK_LPxJI
2,https://www.youtube.com/watch?v=HIEhHAEUJiw,Right,HIEhHAEUJiw
3,https://www.youtube.com/watch?v=_pWNxNq3Nps,Right,_pWNxNq3Nps
4,https://www.youtube.com/watch?v=DSbWRvKyIuE,Mainstream,DSbWRvKyIuE


In [51]:
CHROMEDRIVER_PATH = None  # cached so we don't re-resolve it before every restart

def create_driver(headless=False, block_images=False):
    """Launch a fresh, isolated Chrome instance: incognito + new temp profile dir."""
    options = Options()
    options.add_argument("--incognito")

    profile_dir = tempfile.mkdtemp(prefix="yt_scrape_profile_")
    options.add_argument(f"--user-data-dir={profile_dir}")

    if headless:
        options.add_argument("--headless=new")

    options.add_argument("--no-sandbox")
    options.add_argument("--disable-dev-shm-usage")
    options.add_argument("--window-size=1920,1080")
    options.add_argument("--lang=en-GB")
    options.add_argument("--disable-blink-features=AutomationControlled")
    options.add_experimental_option("excludeSwitches", ["enable-automation"])
    options.add_experimental_option("useAutomationExtension", False)

    if block_images:
        options.add_experimental_option("prefs", {"profile.managed_default_content_settings.images": 2})

    options.add_argument(
        "user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
        "AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36"
    )

    global CHROMEDRIVER_PATH
    if USE_WDM:
        if CHROMEDRIVER_PATH is None:
            CHROMEDRIVER_PATH = ChromeDriverManager().install()
        driver = webdriver.Chrome(service=Service(CHROMEDRIVER_PATH), options=options)
    else:
        driver = webdriver.Chrome(options=options)

    driver.execute_script("Object.defineProperty(navigator, 'webdriver', {get: () => undefined})")
    driver._yt_scrape_profile_dir = profile_dir
    return driver


def quit_driver(d):
    """Close a driver and remove its temp profile dir."""
    profile_dir = getattr(d, "_yt_scrape_profile_dir", None)
    try:
        d.quit()
    except Exception:
        pass
    if profile_dir:
        shutil.rmtree(profile_dir, ignore_errors=True)


driver = create_driver(headless=False)
print("Chrome launched successfully (incognito mode, visible window, fresh temp profile).")

Chrome launched successfully (incognito mode, visible window, fresh temp profile).


In [52]:
def dismiss_cookie_popup(driver, wait_seconds=8):
    """Click through the GDPR/cookie consent popup if present."""
    try:
        btn = WebDriverWait(driver, wait_seconds).until(
            EC.element_to_be_clickable((
                By.XPATH,
                "//button[contains(., 'Accept all') or contains(., 'Reject all') or contains(., 'Accept')]"
            ))
        )
        btn.click()
        time.sleep(1)
        return True
    except TimeoutException:
        return False


driver.get("https://www.youtube.com")
time.sleep(3)

if dismiss_cookie_popup(driver):
    print("Cookie弹窗已处理。")
else:
    print("没有出现cookie弹窗，继续。")

print(f"当前页面: {driver.current_url}")

Cookie弹窗已处理。
当前页面: https://www.youtube.com/


In [ ]:
SIDEBAR_SELECTORS = ["ytd-compact-video-renderer", "yt-lockup-view-model"]

CHANNEL_SELECTORS = [
    ".ytContentMetadataViewModelMetadataRow:first-of-type .ytContentMetadataViewModelMetadataText",
    ".ytContentMetadataViewModelMetadataText",
    "ytd-channel-name #text",
    "ytd-channel-name yt-formatted-string",
    "#channel-name yt-formatted-string",
    "#channel-name",
    "#text.ytd-channel-name",
]

OWN_CHANNEL_SELECTORS = [
    "ytd-video-owner-renderer ytd-channel-name a",
    "#owner ytd-channel-name a",
    "#upload-info #channel-name a",
    "ytd-channel-name#channel-name a",
]


def find_recommendation_elements(driver):
    for selector in SIDEBAR_SELECTORS:
        elements = driver.find_elements(By.CSS_SELECTOR, selector)
        if elements:
            return elements
    return []


def extract_rec_from_element(elem):
    """Extract video_id / title / channel / link from one sidebar recommendation element."""
    href = None
    for sel in ["a#thumbnail", "a[href*='watch?v='], a[href*='/shorts/']"]:
        try:
            href = elem.find_element(By.CSS_SELECTOR, sel).get_attribute("href")
            if href:
                break
        except NoSuchElementException:
            continue
    if not href:
        return None

    rec_id = extract_video_id(href)
    if not rec_id:
        return None

    title = ""
    for sel in ["#video-title", "span#video-title", "h3", "[title]"]:
        try:
            t = elem.find_element(By.CSS_SELECTOR, sel)
            candidate = (t.get_attribute("title") or t.text or "").strip()
            if candidate:
                title = candidate
                break
        except NoSuchElementException:
            continue

    channel = ""
    for sel in CHANNEL_SELECTORS:
        try:
            c = elem.find_element(By.CSS_SELECTOR, sel)
            candidate = (c.text or c.get_attribute("innerText") or "").strip()
            if candidate:
                channel = candidate
                break
        except NoSuchElementException:
            continue

    return {"video_id": rec_id, "title": title, "channel": channel,
            "link": f"https://www.youtube.com/watch?v={rec_id}"}


def get_current_video_title(driver):
    try:
        elem = WebDriverWait(driver, 10).until(
            EC.presence_of_element_located((By.CSS_SELECTOR, "h1.ytd-watch-metadata yt-formatted-string"))
        )
        if elem.text.strip():
            return elem.text.strip()
    except (TimeoutException, NoSuchElementException):
        pass
    try:
        return driver.title.replace(" - YouTube", "").strip()
    except Exception:
        return None


def get_current_channel_name(driver):
    """Channel name of the currently playing video, read from the owner info box."""
    for sel in OWN_CHANNEL_SELECTORS:
        try:
            elem = WebDriverWait(driver, 8).until(EC.presence_of_element_located((By.CSS_SELECTOR, sel)))
            if elem.text.strip():
                return elem.text.strip()
        except (TimeoutException, NoSuchElementException):
            continue
    return None


def scrape_related_videos(driver, video_id, top_n=TOP_N):
    """Visit a video page and return (own_title, own_channel, related[:top_n])."""
    url = f"https://www.youtube.com/watch?v={video_id}"
    related, own_title, own_channel = [], None, None

    try:
        driver.get(url)
        dismiss_cookie_popup(driver, wait_seconds=5)
        WebDriverWait(driver, PAGE_TIMEOUT).until(lambda d: len(find_recommendation_elements(d)) > 0)
        time.sleep(3)

        own_title = get_current_video_title(driver)
        own_channel = get_current_channel_name(driver)

        rank = 1
        for elem in find_recommendation_elements(driver):
            if rank > top_n:
                break
            rec = extract_rec_from_element(elem)
            if rec is None:
                continue
            rec["rank"] = rank
            related.append(rec)
            rank += 1

    except TimeoutException:
        print(f"    [TIMEOUT] {video_id} — no related videos found (age-restricted/private/deleted?)")
    except WebDriverException as e:
        if is_browser_crashed(e):
            raise
        print(f"    [ERROR] {video_id}: {str(e)[:100]}")

    return own_title, own_channel, related


def is_browser_crashed(exception):
    msg = str(exception).lower()
    return any(kw in msg for kw in [
        "tab crashed", "session deleted", "invalid session id",
        "chrome not reachable", "disconnected", "target window already closed",
    ])


def restart_driver(reason="scheduled restart"):
    """Fully close the current Chrome (and its temp profile) and launch a fresh incognito instance."""
    global driver
    quit_driver(driver)
    print(f"    [重启Chrome: {reason}]")
    driver = create_driver(headless=False, block_images=True)
    driver.get("https://www.youtube.com")
    time.sleep(3)
    dismiss_cookie_popup(driver)
    print("    [重启完成，继续抓取]")
    return driver


def scrape_related_videos_safe(video_id, top_n=TOP_N, max_retries=2):
    """scrape_related_videos with auto-restart-and-retry if the browser crashes."""
    global driver
    for attempt in range(max_retries + 1):
        try:
            return scrape_related_videos(driver, video_id, top_n=top_n)
        except WebDriverException as e:
            if is_browser_crashed(e) and attempt < max_retries:
                restart_driver(reason="browser crash")
                time.sleep(3)
                continue
            print(f"    [giving up] {video_id}: still failing after {attempt} retries — {str(e)[:100]}")
            return None, None, []
    return None, None, []


print("抓取函数定义完成。")

抓取函数定义完成。


In [55]:
# Level 1: seed videos -> first-level related videos
all_records = []
seed_list = seed_df.to_dict("records")

print(f"开始抓取 {len(seed_list)} 个seed videos的first-level related videos...")
print(f"预计耗时: {len(seed_list) * (MIN_DELAY + MAX_DELAY) / 2 / 60:.0f} 分钟\n")

for i, row in enumerate(tqdm(seed_list, desc="Level 1")):
    seed_id = row["video_id"]
    seed_stance = row["stance"]

    restart_driver(reason=f"下一个video（seed {i+1}/{len(seed_list)}）")
    print(f"[{i+1}/{len(seed_list)}] Seed: {seed_id} ({seed_stance})")

    own_title, own_channel, related = scrape_related_videos_safe(seed_id, top_n=TOP_N)

    all_records.append({
        "video_id": seed_id, "title": own_title, "channel": own_channel,
        "link": f"https://www.youtube.com/watch?v={seed_id}", "level": "seed",
        "seed_video_id": seed_id, "seed_stance": seed_stance,
        "source_video_id": None, "rank": None,
    })

    for rec in related:
        all_records.append({
            "video_id": rec["video_id"], "title": rec["title"], "channel": rec["channel"],
            "link": rec["link"], "level": "first_level",
            "seed_video_id": seed_id, "seed_stance": seed_stance,
            "source_video_id": seed_id, "rank": rec["rank"],
        })

    print(f"    抓到 {len(related)} 条first-level related videos")

    if (i + 1) % 5 == 0:
        pd.DataFrame(all_records).to_csv(CHECKPOINT_L1, index=False)
        print(f"    [checkpoint saved, {len(all_records)} rows so far]")

    time.sleep(random.uniform(MIN_DELAY, MAX_DELAY))

pd.DataFrame(all_records).to_csv(CHECKPOINT_L1, index=False)
print(f"\nLevel 1 完成。目前共 {len(all_records)} 行记录。")

开始抓取 5 个seed videos的first-level related videos...
预计耗时: 1 分钟



Level 1:   0%|          | 0/5 [00:00<?, ?it/s]

    [重启Chrome: 下一个video（seed 1/5）]
    [重启完成，继续抓取]
[1/5] Seed: HgrQsU8TMmQ (Mainstream)
    抓到 5 条first-level related videos
    [重启Chrome: 下一个video（seed 2/5）]
    [重启完成，继续抓取]
[2/5] Seed: A-zFK_LPxJI (Mainstream)
    抓到 5 条first-level related videos
    [重启Chrome: 下一个video（seed 3/5）]
    [重启完成，继续抓取]
[3/5] Seed: HIEhHAEUJiw (Right)
    抓到 5 条first-level related videos
    [重启Chrome: 下一个video（seed 4/5）]
    [重启完成，继续抓取]
[4/5] Seed: _pWNxNq3Nps (Right)
    抓到 5 条first-level related videos
    [重启Chrome: 下一个video（seed 5/5）]
    [重启完成，继续抓取]
[5/5] Seed: DSbWRvKyIuE (Mainstream)
    抓到 5 条first-level related videos
    [checkpoint saved, 30 rows so far]

Level 1 完成。目前共 30 行记录。


In [56]:
# Level 2: first-level videos -> second-level related videos
first_level_rows = [r for r in all_records if r["level"] == "first_level"]

print(f"first-level记录共 {len(first_level_rows)} 条")
print(f"将逐条访问，共 {len(first_level_rows)} 次页面请求（不做去重）\n")

for i, row in enumerate(tqdm(first_level_rows, desc="Level 2")):
    fl_id = row["video_id"]
    seed_id = row["seed_video_id"]
    seed_stance = row["seed_stance"]

    restart_driver(reason=f"下一个video（first-level {i+1}/{len(first_level_rows)}）")
    print(f"[{i+1}/{len(first_level_rows)}] First-level: {fl_id} (seed={seed_id})")

    _, own_channel, related = scrape_related_videos_safe(fl_id, top_n=TOP_N)
    if own_channel:
        row["channel"] = own_channel

    for rec in related:
        all_records.append({
            "video_id": rec["video_id"], "title": rec["title"], "channel": rec["channel"],
            "link": rec["link"], "level": "second_level",
            "seed_video_id": seed_id, "seed_stance": seed_stance,
            "source_video_id": fl_id, "rank": rec["rank"],
        })

    print(f"    抓到 {len(related)} 条second-level related videos")

    if (i + 1) % 10 == 0:
        pd.DataFrame(all_records).to_csv(CHECKPOINT_L2, index=False)
        print(f"    [checkpoint saved, {len(all_records)} rows so far]")

    time.sleep(random.uniform(MIN_DELAY, MAX_DELAY))

pd.DataFrame(all_records).to_csv(CHECKPOINT_L2, index=False)
print(f"\\nLevel 2 完成。共发起了 {len(first_level_rows)} 次页面请求。")
print(f"目前共 {len(all_records)} 行记录。")

first-level记录共 25 条
将逐条访问，共 25 次页面请求（不做去重）



Level 2:   0%|          | 0/25 [00:00<?, ?it/s]

    [重启Chrome: 下一个video（first-level 1/25）]
    [重启完成，继续抓取]
[1/25] First-level: QsLw0c1SiFQ (seed=HgrQsU8TMmQ)
    抓到 5 条second-level related videos
    [重启Chrome: 下一个video（first-level 2/25）]
    [重启完成，继续抓取]
[2/25] First-level: nL-nDRlUzfY (seed=HgrQsU8TMmQ)
    抓到 5 条second-level related videos
    [重启Chrome: 下一个video（first-level 3/25）]
    [重启完成，继续抓取]
[3/25] First-level: EK1PsvorDm0 (seed=HgrQsU8TMmQ)
    抓到 5 条second-level related videos
    [重启Chrome: 下一个video（first-level 4/25）]
    [重启完成，继续抓取]
[4/25] First-level: fQr3W_l15-Q (seed=HgrQsU8TMmQ)
    抓到 5 条second-level related videos
    [重启Chrome: 下一个video（first-level 5/25）]
    [重启完成，继续抓取]
[5/25] First-level: AC1oVk79HtU (seed=HgrQsU8TMmQ)
    抓到 5 条second-level related videos
    [重启Chrome: 下一个video（first-level 6/25）]
    [重启完成，继续抓取]
[6/25] First-level: x3eDp0usbJI (seed=A-zFK_LPxJI)
    抓到 5 条second-level related videos
    [重启Chrome: 下一个video（first-level 7/25）]
    [重启完成，继续抓取]
[7/25] First-level: Uw99HQSDkmY (seed=A-zFK_LPxJI)
  

In [57]:
quit_driver(driver)
print("Browser closed.")

Browser closed.


In [58]:
# columns: video_id, title, channel, link, level (seed/first_level/second_level),
# seed_video_id, seed_stance, source_video_id, rank
final_df = pd.DataFrame(all_records)
final_df.to_csv(OUTPUT_FILE, index=False)

print(f"已保存到: {OUTPUT_FILE}")
print(f"总行数: {len(final_df)}\n")

print("按level统计:")
print(final_df["level"].value_counts())
print()
print("按seed_stance x level统计:")
print(final_df.groupby(["seed_stance", "level"]).size())

已保存到: restart_recomm_4/restart_recom_4.csv
总行数: 155

按level统计:
level
second_level    125
first_level      25
seed              5
Name: count, dtype: int64

按seed_stance x level统计:
seed_stance  level       
Mainstream   first_level     15
             second_level    75
             seed             3
Right        first_level     10
             second_level    50
             seed             2
dtype: int64


In [59]:
print(f"Unique videos总数（去重后）: {final_df['video_id'].nunique()}")
print(f"Unique seed videos: {final_df['seed_video_id'].nunique()}")
print()

counts_per_parent = (
    final_df[final_df["level"] != "seed"]
    .groupby(["source_video_id", "level"])
    .size()
)
incomplete = counts_per_parent[counts_per_parent < TOP_N]
if len(incomplete) > 0:
    print(f"{len(incomplete)} parent videos have fewer than {TOP_N} related videos:")
    print(incomplete)
else:
    print(f"所有父视频都抓到了完整的{TOP_N}条related videos。")

print()
final_df.head(10)

Unique videos总数（去重后）: 107
Unique seed videos: 5

所有父视频都抓到了完整的5条related videos。



,video_id,title,channel,link,level,seed_video_id,seed_stance,source_video_id,rank
0,HgrQsU8TMmQ,"A total of 41,472 migrants arrived in the UK i...",ITV News,https://www.youtube.com/watch?v=HgrQsU8TMmQ,seed,HgrQsU8TMmQ,Mainstream,None,NaN
1,QsLw0c1SiFQ,When Tyson Faced the Smash Machine,VS+,https://www.youtube.com/watch?v=QsLw0c1SiFQ,first_level,HgrQsU8TMmQ,Mainstream,HgrQsU8TMmQ,1.0
2,nL-nDRlUzfY,WORLD EXCLUSIVE: Prince Harry's Invictus Games...,Daily Express,https://www.youtube.com/watch?v=nL-nDRlUzfY,first_level,HgrQsU8TMmQ,Mainstream,HgrQsU8TMmQ,2.0
3,EK1PsvorDm0,James Webb's Red Dots Have an Answer — And It'...,Late Science,https://www.youtube.com/watch?v=EK1PsvorDm0,first_level,HgrQsU8TMmQ,Mainstream,HgrQsU8TMmQ,3.0
4,fQr3W_l15-Q,THE PIE CHALLENGE THAT NOBODY HAS MANAGED TO C...,BeardMeatsFood,https://www.youtube.com/watch?v=fQr3W_l15-Q,first_level,HgrQsU8TMmQ,Mainstream,HgrQsU8TMmQ,4.0
5,AC1oVk79HtU,You have to hear what Trump just said,David Pakman Show,https://www.youtube.com/watch?v=AC1oVk79HtU,first_level,HgrQsU8TMmQ,Mainstream,HgrQsU8TMmQ,5.0
6,A-zFK_LPxJI,'Nigel Farage can sod off' says Home Secretary...,ITV News,https://www.youtube.com/watch?v=A-zFK_LPxJI,seed,A-zFK_LPxJI,Mainstream,None,NaN
7,x3eDp0usbJI,'BRACE YOURSELVES!' Peter Bleksley DESTROYS An...,GBNews,https://www.youtube.com/watch?v=x3eDp0usbJI,first_level,A-zFK_LPxJI,Mainstream,A-zFK_LPxJI,1.0
8,Uw99HQSDkmY,"Laila Cunningham, Hannah Spencer & Fraser Nels...",BBC Politics,https://www.youtube.com/watch?v=Uw99HQSDkmY,first_level,A-zFK_LPxJI,Mainstream,A-zFK_LPxJI,2.0
9,RzsgKgKOqVM,🚔 Stopped by the Police? There's Only ONE Thin...,Oliver Bennett,https://www.youtube.com/watch?v=RzsgKgKOqVM,first_level,A-zFK_LPxJI,Mainstream,A-zFK_LPxJI,3.0


In [62]:
# Merge multiple batch runs into one file
files = [
    "restart_recom_1.csv",
    "restart_recom_2.csv",
    "restart_recom_3.csv",
    "restart_recom_4.csv",
]

dfs = [pd.read_csv(f) for f in files]
merged = pd.concat(dfs, ignore_index=True)
print(f"合并前总行数: {len(merged)}")

merged = merged.drop_duplicates()
print(f"去重后总行数: {len(merged)}")

key_cols = ["level", "seed_video_id", "source_video_id", "video_id", "rank"]
conflict_mask = merged.duplicated(subset=key_cols, keep=False)
conflicts = merged[conflict_mask].sort_values(key_cols)

if len(conflicts) > 0:
    print(f"\n{conflicts[key_cols].drop_duplicates().shape[0]} groups of records have mismatched content, please check:")
    print(conflicts[key_cols + ["title", "channel"]])
else:
    print("\n没有发现内容冲突的重复记录，可以放心用。")

merged.to_csv("two_level_recommendations_merged.csv", index=False)
print(f"\n已保存到 two_level_recommendations_merged.csv，共 {len(merged)} 行")

print("\n按level统计:")
print(merged["level"].value_counts())
print(f"\nUnique videos: {merged['video_id'].nunique()}")
print(f"Unique seeds: {merged['seed_video_id'].nunique()}")

合并前总行数: 620
去重后总行数: 620

没有发现内容冲突的重复记录，可以放心用。

已保存到 two_level_recommendations_merged.csv，共 620 行

按level统计:
level
second_level    500
first_level     100
seed             20
Name: count, dtype: int64

Unique videos: 339
Unique seeds: 20
